In [1]:
import json
from scripts.parsers import parse_sequences
from utils.video_tools import get_camera_calibration_files
from utils.multicamera_tools import parse_camera_xml, triangulate_poses

sequences_file_path = 'gait3d/ListOfSequences.txt'
sequences = parse_sequences(sequences_file_path)
sequences['p1s1']

{'start_frame': 195,
 'number_of_frames': 135,
 'frame_offset': 0,
 'MoCap_data': True}

In [4]:
datasets = {
    "movenet_lightning": "datasets/movenet/dataset_with_confidence_lightning.json",
    "movenet_thunder": "datasets/movenet/dataset_with_confidence_thunder.json",
}


In [5]:
selected_joints_file = './datasets/yolo/selected_joint_names_v2.json'

with open(selected_joints_file, 'r') as file:
    selected_joints = json.load(file)

coco_joints = {int(num): joint_name for num, joint_name in selected_joints.items()}
coco_joints

{5: 'lhumerus',
 6: 'rhumerus',
 11: 'lfemur',
 12: 'rfemur',
 13: 'ltibia',
 14: 'rtibia',
 15: 'lfoot',
 16: 'rfoot',
 7: 'lradius',
 8: 'rradius',
 9: 'lwrist',
 10: 'rwrist'}

In [25]:
import numpy as np

FRAME_SIZE = [960, 540]
SCALE_FACTOR = 1/255
CAMERAS = ['c1', 'c2', 'c3', 'c4']

def triangulate_yolo_movenet_with_confidence(dataset_path: str):
    with open(dataset_path, 'r') as file:
        dataset = json.load(file)

    dataset_triangulated = {}

    for seq_key in dataset.keys():
        print(f"| {seq_key} ", end = '')
        
        squence_2d = dataset[seq_key]
        expected_frames_num = sequences[seq_key]['number_of_frames']
        
        triangulation_results = []
        
        camera_files_paths = get_camera_calibration_files(seq_key)
        cameras_params = [parse_camera_xml(camera_path) for camera_path in camera_files_paths]
        
        if all([expected_frames_num == len(joints_2d) for joints_2d in squence_2d.values()]):
            for f_idx in range(expected_frames_num):
                frame_triangulation_result = {}
                for joint_idx, joint_name in coco_joints.items():
                    confidences = {camera:squence_2d[camera][f_idx][joint_idx][2] for camera in CAMERAS}
                    min_confidence = min(confidences.values())
                    found_for_cameras = {camera:(confidences[camera] != min_confidence) for camera in CAMERAS}
                    # select only views for which lanmarks are not []
                    joint_2d = np.array([np.array([squence_2d[camera][f_idx][joint_idx][:2]]) * FRAME_SIZE for camera in CAMERAS if found_for_cameras[camera]])
                    selected_cameras_params =  [cameras_params[i] for i, found in enumerate(found_for_cameras.values()) if found]
                    
                    joint_triangulation_result = SCALE_FACTOR*triangulate_poses(selected_cameras_params, joint_2d)
                    frame_triangulation_result[joint_name] = joint_triangulation_result[0][0].tolist()        
                triangulation_results.append(frame_triangulation_result)

        dataset_triangulated[seq_key] = triangulation_results

    destination_file = dataset_path.replace('.json', '_triangulated_best_cameras.json')
    with open(destination_file, 'w') as f:
        json.dump(dataset_triangulated, f, indent=4)

    return dataset_triangulated

In [26]:
_ = triangulate_yolo_movenet_with_confidence(datasets["movenet_lightning"])

| p1s1 | p1s2 | p1s3 | p1s4 | p2s1 | p2s2 | p2s3 | p2s4 | p3s1 | p3s2 | p3s3 | p3s4 | p4s1 | p4s2 | p4s3 | p4s4 | p5s1 | p5s2 | p5s3 | p5s4 | p6s1 | p6s2 | p6s3 | p6s4 | p7s1 | p7s2 | p7s3 | p7s4 | p8s1 | p8s2 | p8s3 | p8s4 | p9s1 | p9s2 | p9s3 | p9s4 | p10s1 | p10s2 | p10s3 | p10s4 | p11s1 | p11s2 | p11s3 | p11s4 | p12s1 | p12s2 | p12s3 | p12s4 | p13s1 | p13s2 | p13s3 | p13s4 | p14s1 | p14s2 | p14s3 | p14s4 | p15s1 | p15s2 | p15s3 | p15s4 | p16s1 | p16s2 | p16s3 | p16s4 | p17s1 | p17s2 | p17s3 | p17s4 | p18s1 | p18s2 | p18s3 | p18s4 | p19s1 | p19s2 | p19s3 | p19s4 | p20s1 | p20s2 | p20s3 | p20s4 | p21s1 | p21s2 | p21s3 | p21s4 | p22s1 | p22s2 | p22s3 | p22s4 | p23s1 | p23s2 | p23s3 | p23s4 | p24s1 | p24s2 | p24s3 | p24s4 | p25s1 | p25s2 | p25s3 | p25s4 | p26s1 | p26s2 | p26s3 | p26s4 | p26s5 | p26s6 | p26s7 | p26s8 | p26s9 | p26s10 | p27s1 | p27s2 | p27s3 | p27s4 | p27s5 | p27s6 | p27s7 | p27s8 | p27s9 | p27s10 | p28s1 | p28s2 | p28s3 | p28s4 | p28s5 | p28s6 | p28s7 | p28s8 | p28s9 | 

In [27]:
_ = triangulate_yolo_movenet_with_confidence(datasets["movenet_thunder"])

| p1s1 | p1s2 | p1s3 | p1s4 | p2s1 | p2s2 | p2s3 | p2s4 | p3s1 | p3s2 | p3s3 | p3s4 | p4s1 | p4s2 | p4s3 | p4s4 | p5s1 | p5s2 | p5s3 | p5s4 | p6s1 | p6s2 | p6s3 | p6s4 | p7s1 | p7s2 | p7s3 | p7s4 | p8s1 | p8s2 | p8s3 | p8s4 | p9s1 | p9s2 | p9s3 | p9s4 | p10s1 | p10s2 | p10s3 | p10s4 | p11s1 | p11s2 | p11s3 | p11s4 | p12s1 | p12s2 | p12s3 | p12s4 | p13s1 | p13s2 | p13s3 | p13s4 | p14s1 | p14s2 | p14s3 | p14s4 | p15s1 | p15s2 | p15s3 | p15s4 | p16s1 | p16s2 | p16s3 | p16s4 | p17s1 | p17s2 | p17s3 | p17s4 | p18s1 | p18s2 | p18s3 | p18s4 | p19s1 | p19s2 | p19s3 | p19s4 | p20s1 | p20s2 | p20s3 | p20s4 | p21s1 | p21s2 | p21s3 | p21s4 | p22s1 | p22s2 | p22s3 | p22s4 | p23s1 | p23s2 | p23s3 | p23s4 | p24s1 | p24s2 | p24s3 | p24s4 | p25s1 | p25s2 | p25s3 | p25s4 | p26s1 | p26s2 | p26s3 | p26s4 | p26s5 | p26s6 | p26s7 | p26s8 | p26s9 | p26s10 | p27s1 | p27s2 | p27s3 | p27s4 | p27s5 | p27s6 | p27s7 | p27s8 | p27s9 | p27s10 | p28s1 | p28s2 | p28s3 | p28s4 | p28s5 | p28s6 | p28s7 | p28s8 | p28s9 | 

In [28]:
datasets = {
    "openpose": "datasets/openpose/dataset_with_confidence_openpose.json",
}

body_25_joints = {
    14 : "lfoot", 
    11 : "rfoot", 
    13 : "ltibia", 
    10 : "rtibia", 
    12 : "lfemur", 
    9 : "rfemur",
    5 : "lhumerus", 
    2 : "rhumerus",
    6 : "lradius", 
    3 : "rradius",
    7 : "lwrist", 
    4 : "rwrist",
}

body_25_joints

{14: 'lfoot',
 11: 'rfoot',
 13: 'ltibia',
 10: 'rtibia',
 12: 'lfemur',
 9: 'rfemur',
 5: 'lhumerus',
 2: 'rhumerus',
 6: 'lradius',
 3: 'rradius',
 7: 'lwrist',
 4: 'rwrist'}

In [29]:
import numpy as np

FRAME_SIZE = [960, 540]
SCALE_FACTOR = 1/255
CAMERAS = ['c1', 'c2', 'c3', 'c4']

def triangulate_openpose_with_confidence(dataset_path: str):
    with open(dataset_path, 'r') as file:
        dataset = json.load(file)

    dataset_triangulated = {}

    for seq_key in dataset.keys():
        print(f"| {seq_key} ", end = '')
        
        squence_2d = dataset[seq_key]
        expected_frames_num = sequences[seq_key]['number_of_frames']
        
        triangulation_results = []
        
        camera_files_paths = get_camera_calibration_files(seq_key)
        cameras_params = [parse_camera_xml(camera_path) for camera_path in camera_files_paths]
        
        if all([expected_frames_num == len(joints_2d) for joints_2d in squence_2d.values()]):
            for f_idx in range(expected_frames_num):
                frame_triangulation_result = {}
                for joint_idx, joint_name in body_25_joints.items():
                    found_for_cameras = {camera:(squence_2d[camera][f_idx][joint_idx] if squence_2d[camera][f_idx] else False) for camera in CAMERAS}
                    if all(found_for_cameras.values()):
                        min_confidence = 1
                        for parameters in found_for_cameras.values():
                            min_confidence = min(min_confidence, parameters[2])
 
                        found_for_cameras = {camera:(squence_2d[camera][f_idx][joint_idx][2] != min_confidence) for camera in CAMERAS}

                    joint_2d = np.array([np.array([squence_2d[camera][f_idx][joint_idx][:2]]) * FRAME_SIZE for camera in CAMERAS if found_for_cameras[camera]])
                    selected_cameras_params =  [cameras_params[i] for i, found in enumerate(found_for_cameras.values()) if found]
                    
                    joint_triangulation_result = SCALE_FACTOR*triangulate_poses(selected_cameras_params, joint_2d)
                    frame_triangulation_result[joint_name] = joint_triangulation_result[0][0].tolist()        
                triangulation_results.append(frame_triangulation_result)

        dataset_triangulated[seq_key] = triangulation_results

    destination_file = dataset_path.replace('.json', '_triangulated_best_cameras.json')
    with open(destination_file, 'w') as f:
        json.dump(dataset_triangulated, f, indent=4)

    return dataset_triangulated

In [ ]:
_ = triangulate_openpose_with_confidence(datasets["openpose"])